# TUTORIAL 04: ADVANCED TDA AND THE ULTIMATE PIPELINE

Up until now, we provided the mathematical boundary matrices or intersection forms manually. We typed `np.array([[0, 1], [1, 0]])` because we already knew the manifold was $S^2 \times S^2$.

But in the real world, we deal with **raw, discrete data** (like point clouds from 3D/4D scans, or high-dimensional word embeddings), and we have **no idea** what manifold the data represents.

This tutorial explains how `pysurgery` completely bridges **Topological Data Analysis (TDA)** and Surgery Theory to automatically discover the exact homeomorphism type of a raw point cloud.

## 1. The Problem with Discrete Data

When you have points, you build a Simplicial Complex (like a Vietoris-Rips or Alpha Complex). Ecosystem software like `gudhi` easily computes Betti numbers (free ranks) from these complexes. 

But as we learned in Tutorial 2, Betti numbers are **NOT enough** in 4D. To run Freedman's classification or Surgery, we absolutely need the **Intersection Form matrix $Q$**.
And you cannot find the Intersection Form directly from boundaries. You need the **Cup Product**.

## 2. The Alexander-Whitney Cup Product
The Cup Product $\smile: H^2(M) \times H^2(M) \to H^4(M)$ is the operation that allows us to multiply cohomology classes. By evaluating this product on the fundamental class $[M] \in H_4$, we recover the exact symmetric integer matrix $Q$.

Our `integrations.gudhi_bridge` automates this massive mathematical translation:
1. **Extraction**: It reads the raw simplices (triangles, tetrahedrons) from a `gudhi.SimplexTree`.
2. **Cohomology Basis**: It computes the exact integer nullspace of the boundary operators to find the $H^2$ cochain basis.
3. **Alexander-Whitney Formula**: It evaluates the formula $(\alpha \smile \beta)([v_0, v_1, v_2, v_3, v_4]) = \alpha([v_0, v_1, v_2]) \cdot \beta([v_2, v_3, v_4])$ on every single discrete 4-simplex in the dataset.
4. **Integration**: It sums these values over the fundamental class $[M]$ to output the `IntersectionForm`.

In [1]:
import numpy as np
from pysurgery.integrations.gudhi_bridge import simplex_tree_to_intersection_form
from pysurgery.homeomorphism import analyze_homeomorphism_4d
from pysurgery.core.intersection_forms import IntersectionForm

try:
    import gudhi
    HAS_GUDHI = True
    print("GUDHI detected. We can process discrete TDA data.")
except ImportError:
    HAS_GUDHI = False
    print("Please install gudhi: pip install gudhi")

/home/gabriel/Desktop/SurgeryTheory/pysurgery/core/complexes.py:69: SyntaxWarning: invalid escape sequence '\c'
  """


GUDHI detected. We can process discrete TDA data.


## 3. Creating a Discrete Manifold

To test this pipeline, we need a 4D manifold. Generating a full Vietoris-Rips Complex for a 4D point cloud generates millions of 5-dimensional simplices, which requires the Julia Sparse backend and takes too long for a quick notebook.

Instead, we will mathematically synthesize a discrete 4-Sphere $S^4$. The hollow boundary of a 5-simplex is topologically equivalent to $S^4$. We will manually inject these combinatorial edges, faces, and tetrahedrons into GUDHI to simulate the output of a TDA filtration.

In [2]:
if HAS_GUDHI:
    st = gudhi.SimplexTree()
    # A 5-simplex has 6 vertices: [0, 1, 2, 3, 4, 5]
    # It represents a solid 5D ball.
    st.insert([0, 1, 2, 3, 4, 5])
    
    # To get the hollow boundary (which is S^4), we remove the interior solid cell.
    # We are left with only the boundary 4-simplices.
    st.remove_maximal_simplex([0, 1, 2, 3, 4, 5])
    
    print("Constructed discrete SimplexTree representing S^4.")
    print(f"Number of simplices in the tree: {st.num_simplices()}")

Constructed discrete SimplexTree representing S^4.
Number of simplices in the tree: 62


## 4. The Automated Pipeline
Now we feed this completely opaque combinatorial data structure into `pysurgery`. It has no prior knowledge that this is a sphere. It only sees a soup of vertices and edges.

In [3]:
if HAS_GUDHI:
    print("Extracting Intersection Form via Alexander-Whitney Cup Product...")
    # This one line performs SNF, Nullspace extraction, and combinatorial cup products!
    q_form = simplex_tree_to_intersection_form(st)
    
    print("\n--- AUTOMATED EXTRACTION COMPLETE ---")
    print(f"Derived Matrix:\n{q_form.matrix}")
    print(f"Derived Rank: {q_form.rank()}")
    print(f"Derived Signature: {q_form.signature()}")
    print("Notice the rank is 0. The pipeline successfully deduced that this shape has no 2D holes.")

Extracting Intersection Form via Alexander-Whitney Cup Product...

--- AUTOMATED EXTRACTION COMPLETE ---
Derived Matrix:
[]
Derived Rank: 0
Derived Signature: 0
Notice the rank is 0. The pipeline successfully deduced that this shape has no 2D holes.


## 5. Mathematical Homeomorphism Verification
We found that the matrix is empty (Rank 0). We can now ask `pysurgery` to compare this data-derived form against the mathematical ideal of a 4-Sphere, using Freedman's theorem.

In [4]:
if HAS_GUDHI:
    # Target: Mathematical S^4 (Rank 0 matrix)
    target_s4 = IntersectionForm(matrix=np.zeros((0,0), dtype=int), dimension=4)
    
    is_homeo, report = analyze_homeomorphism_4d(q_form, target_s4)
    
    print(f"Result of Homeomorphism Check:\n{report}")
    
    print("\n>>> PIPELINE COMPLETE! <<<")
    print("We successfully proved that a discrete soup of triangles")
    print("is homeomorphic to a 4-dimensional mathematical sphere.")

Result of Homeomorphism Check:
SUCCESS: Homeomorphism established (assuming lattice isomorphism for these definite forms).

>>> PIPELINE COMPLETE! <<<
We successfully proved that a discrete soup of triangles
is homeomorphic to a 4-dimensional mathematical sphere.
